# 382. Linked List Random Node

**Difficulty:** Medium &nbsp;|&nbsp; **Topics:** linked-list, math, reservoir-sampling, randomized
&nbsp;|&nbsp; [LeetCode](https://leetcode.com/problems/linked-list-random-node/)

Given a singly linked list, return **a random node's value** from the list.
Each node must have the **same probability** of being chosen.

Implement the `Solution` class:

- `Solution(head)` initializes the object with the head of the singly linked
  list `head`.
- `getRandom()` chooses a node randomly from the list and returns its value.
  All the nodes of the list should be **equally likely** to be chosen.

---

### Example 1


```
Input:  ["Solution", "getRandom", "getRandom", "getRandom", "getRandom", "getRandom"]
        [[[1, 2, 3]], [],         [],          [],          [],          []]
Output: [null,        1,          3,           2,           2,           3]

Solution solution = new Solution([1, 2, 3]);
solution.getRandom();  // return 1
solution.getRandom();  // return 3
solution.getRandom();  // return 2
solution.getRandom();  // return 2
solution.getRandom();  // return 3
// getRandom() should return either 1, 2, or 3 randomly.
// Each element should have equal probability of returning.
```

---

### Constraints

- The number of nodes in the linked list will be in the range `[1, 10^4]`.
- `-10^4 <= Node.val <= 10^4`
- At most `10^4` calls will be made to `getRandom`.

### Follow-up

**What if the linked list is extremely large and its length is unknown to
you? Could you solve this efficiently without using extra space?**

That sentence is the whole problem. Route A below ignores it and still gets
accepted; route B answers it, and is the reason this problem is famous.


## Before you write anything

Your first **randomized** problem. Nothing here is about traversal - you can
already walk a linked list in your sleep. It is about a promise you cannot
see: *every node, exactly the same chance*. Work these on paper.

**1.** Warm-up, no linked list. If the values were sitting in a Python list
of length `n`, what probability must each element have? Which one call from
the `random` module gives you that? (You want an index, not a shuffle.)

**2.** Now read the follow-up again: the length is **unknown** and you may not
store the list. So you get **one walk**, and you may keep only **one** value -
your current candidate. Stand on node number `i` (counting from 1). You have
seen `i` nodes and nothing else. If the list ended right here, each of those
`i` nodes must have probability `1/i`. So: with what probability must the node
you are standing on *become* the candidate, replacing the one you were
holding? Write the number, not the code.

**3.** Prove your answer for a 3-node list by hand. Node 1 becomes the
candidate at step 1, then must **survive** steps 2 and 3. Fill this in:

```
P(node 1 wins) = 1        x  (1 - ?)  x  (1 - ?)  = ?
P(node 2 wins) =     ?    x            (1 - ?)  = ?
P(node 3 wins) =                  ?              = ?
```

All three must come out to exactly `1/3`. If they do, you have derived
**reservoir sampling** yourself.

**4.** Generalize: multiply `(1/j)` by the chance of surviving every later
step, `(1 - 1/i)` for `i = j+1 .. n`. Write `(1 - 1/i)` as a single fraction
and watch the product collapse. What is left?

**5.** `getRandom` is called up to `10^4` times. Route A pays `O(n)` **once**;
route B pays `O(n)` **per call** but stores nothing. For `k` calls on `n`
nodes, write both totals. At what point does the trade actually favour B -
and what does the follow-up assume about `n` that makes it the right answer
anyway?

**6.** How do you **test** a random function? You cannot assert an exact
return value. What CAN you assert about 60,000 calls? Two traps to think
about before you look at the test cell: a tolerance too tight makes a correct
solution fail sometimes (a *flaky* test - the worst kind), and one too loose
lets a biased solution pass. What third thing must you do so the test gives
the same answer every run?

**7.** `getRandom` returns a **value**, not a node. If the list is
`[1, 1, 2]`, what fraction of calls should return `1`? Is that still uniform
over *nodes*?


## Two routes - A gets accepted, B answers the follow-up

**A - snapshot the values** *(write this first)*
Walk the list once in `__init__`, copy every value into a Python list, and
let `getRandom` pick a random index. `O(n)` time and `O(n)` space once,
then **`O(1)` per call**. This is accepted, and if `getRandom` is called far
more often than the list changes, it is genuinely the better engineering
choice. It also fails the follow-up completely: it needs the length, and it
stores the whole list.

**B - reservoir sampling** *(the real answer)*
Keep one candidate value and a counter. Walk the list; at node `i` (1-based),
replace the candidate with probability `1/i`. Return the candidate at the
end. `O(n)` time per call, **`O(1)` space**, and it never needs to know the
length in advance - it works on a stream you cannot rewind or measure. This
is the k=1 case of an algorithm that samples `k` items from an unbounded
stream, and it is a genuine interview favourite.

Two ways to flip a `1/i` coin, both fine:
`random.randint(1, i) == 1` or `random.random() < 1 / i`.

Watch the fence-post: at the **first** node the probability must be `1/1`,
i.e. the candidate is always taken. Check that your counter starts where you
think it does - a shifted index gives the head only half its due, and that is
exactly the kind of bias the test cell below is built to catch.

Write route A, run the tests, then rewrite `getRandom` as route B and run the
same tests again - they judge behaviour, not implementation.


In [1]:
import random
from math import sqrt


class ListNode:
    """LeetCode's node for this problem."""
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


def build(values):
    """Build a linked list from a Python list, return its head."""
    head = None
    for v in reversed(values):
        head = ListNode(v, head)
    return head


In [ ]:
class Solution:

    def __init__(self, head):
    def getRandom(self) -> int:


### The test harness

You cannot assert an exact return value, so these helpers assert a
**distribution** instead: call `getRandom` many times, count how often each
value comes back, and check every count sits within a few standard deviations
of `trials / n`.

`sigmas=5` is a deliberate choice - question 6's two traps. Binomial noise
past 5 sigma is about a one-in-a-million fluke, so a correct solution will
not fail by bad luck, while the bias from a shifted counter is hundreds of
sigmas out and cannot slip through. The seed makes every run identical.

Run this cell; don't edit it.


In [ ]:
def check_uniform(values, trials=60000, sigmas=5, seed=42):
    """True if every value shows up trials/n times, within `sigmas` of binomial noise."""
    random.seed(seed)
    sol = Solution(build(values))
    counts = {}
    for _ in range(trials):
        v = sol.getRandom()
        counts[v] = counts.get(v, 0) + 1

    n = len(values)
    p = 1 / n
    expected = trials * p
    allowed = sigmas * sqrt(trials * p * (1 - p))
    stray = sorted(v for v in counts if v not in values)
    worst = max(abs(counts.get(v, 0) - expected) for v in values)
    return (not stray and worst <= allowed), worst, allowed, stray, counts


def show(values, trials=60000, seed=42):
    """Print the observed distribution next to the ideal one."""
    ok, worst, allowed, stray, counts = check_uniform(values, trials, seed=seed)
    expected = trials / len(values)
    print(f"list {values}   {trials} calls   ideal {expected:.0f} each")
    for v in values:
        got = counts.get(v, 0)
        bar = "#" * round(40 * got / (2 * expected)) if expected else ""
        print(f"  {v:>6} : {got:>6}  ({got / trials:6.2%})  {bar}")
    if stray:
        print(f"  !! returned values that are NOT in the list: {stray}")
    print(f"  worst deviation {worst:.0f}, allowed {allowed:.0f} -> {'OK' if ok else 'BIASED'}")


In [ ]:
# tests
CASES = [
    ("the LeetCode example", [1, 2, 3]),
    ("single node - getRandom must always return it", [7]),
    ("five nodes", [1, 2, 3, 4, 5]),
    ("negatives and zero", [-5, 0, 5, 10]),
    ("20 nodes", list(range(20))),
    ("100 nodes", list(range(100))),
]

for name, values in CASES:
    ok, worst, allowed, stray, _ = check_uniform(values)
    print(f"{'OK  ' if ok else 'FAIL'} {name:45} worst dev {worst:8.1f}  allowed {allowed:7.1f}")
    if stray:
        print(f"     !! values not in the list: {stray}")

# question 7: duplicate values are two nodes, so the VALUE shows up twice as often
ok, _, _, _, counts = check_uniform([1, 1, 2])
share = counts.get(1, 0) / 60000
print(f"\n[1,1,2] -> value 1 came back {share:.1%} of the time (uniform over NODES means ~66.7%)")
print("     ", "OK  " if 0.64 < share < 0.69 else "FAIL", "- two of the three nodes hold a 1")

# see it, do not just trust the pass/fail
print()
show([1, 2, 3])


## After it passes

- **Re-run with a different seed.** Change `seed=42` to anything else in a
  scratch cell and confirm route B still passes. A correct sampler survives
  every seed; if yours only passes on 42, the tolerance was hiding something.
- **Answer question 5 with numbers.** For `n = 10^4` nodes and `k = 10^4`
  calls, write the total work for route A and route B. Now say which one you
  would ship, and which one you would say out loud in an interview - they are
  not the same answer, and knowing why is the point.
- **The pattern is bigger than this problem.** Reservoir sampling picks `k`
  items from a stream of unknown length in `O(k)` space: keep the first `k`,
  then at item `i` replace a random one of your `k` with probability `k/i`.
  Same telescoping proof. It is how you sample logs you cannot fit in memory.
- **Where else did "decide as you go" beat "collect then decide" this week?**
  #155 kept a running minimum instead of rescanning; #355's clock stamped
  each tweet at arrival instead of reconstructing order later. Same instinct.
- Siblings: #398 Random Pick Index (reservoir sampling with a filter), #528
  Random Pick with Weight (prefix sums + binary search), #384 Shuffle an
  Array (the Fisher-Yates shuffle, the other algorithm everyone gets subtly
  wrong).
